# Held-Out Evaluation: Base Agent vs Skills-Enhanced Agent

This notebook runs a controlled comparison of both agents on a held-out evaluation dataset,
using the optimized prompt from `06-PromptOptimization.ipynb` as the shared system prompt.

Both agents use the **identical optimized prompt** (loaded from `@production`), so the only
variable is the presence of agent skills. This isolates the incremental contribution of skills
from the gains achieved by prompt optimization alone.

The evaluation uses `mlflow.genai.evaluate()` with the aligned judge from `05-JudgeAlignment.ipynb`,
and results are logged directly to the MLflow experiment.

**Prerequisites:**
- `03_create_agent_definition.ipynb` has written `agent.py`
- `04-Evaluation.ipynb` has created the evaluation dataset
- `05-JudgeAlignment.ipynb` has produced the aligned judge
- `06-PromptOptimization.ipynb` has optimized and registered the prompt to `@production`
- `07-AgentSkillsGeneration.ipynb` has generated skills
- `08_create_agent_with_skills.ipynb` has written `agent_with_skills.py`

In [ ]:
import json as _bootstrap_json
from datetime import datetime as _bootstrap_datetime, timezone as _bootstrap_timezone
EVAL09_STATUS_DIR = "/Volumes/main/at_bat_assistant/agent_skills_gepa/eval09"
def _eval09_status(stage, message, **data):
    payload = {"ts": _bootstrap_datetime.now(_bootstrap_timezone.utc).isoformat(), "stage": stage, "message": message, "data": data}
    dbutils.fs.mkdirs(EVAL09_STATUS_DIR)
    dbutils.fs.put(f"{EVAL09_STATUS_DIR}/status.json", _bootstrap_json.dumps(payload, indent=2), True)
    dbutils.fs.put(f"{EVAL09_STATUS_DIR}/log.jsonl", _bootstrap_json.dumps(payload) + "\n", True)
    print(payload)
_eval09_status("bootstrap", "Starting 09-Evaluation notebook before dependency install")


In [ ]:
%pip install -q --prefer-binary "typing_extensions>=4.15.0" "mlflow>=3.11.1" databricks-agents databricks-mcp databricks-langchain langgraph langgraph-checkpoint-postgres "psycopg[binary,pool]" backoff


In [ ]:
import json
_eval09_status("setup", "Starting Python imports and config load")
import sys
sys.modules.pop("typing_extensions", None)
import os
import hashlib
import importlib
import warnings
import logging
from contextlib import contextmanager
from pathlib import Path

import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())

# Match the Free Edition deployment path used by notebooks 03 and 08.
os.environ.setdefault("DISABLE_LAKEBASE", "true")
os.environ.setdefault("DISABLE_VECTOR_TOOLS", "true")
os.environ.setdefault("USE_MCP_FUNCTION_EXECUTION", "false")
os.environ.setdefault("UC_SQL_TOOL_TIMEOUT_SECONDS", "120")
os.environ.setdefault("LANGGRAPH_RECURSION_LIMIT", "25")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "300")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "1")

EXPERIMENT_ID = CONFIG["mlflow"]["experiment_id"]
PROMPT_NAME = CONFIG["prompt_registry"]["prompt_name"]
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]

JUDGE_EXPERIMENT_ID = EXPERIMENT_ID

_scope = CONFIG["prompt_registry_auth"]["secret_scope_name"]
_sp_id = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_id_key"])
_sp_secret = dbutils.secrets.get(scope=_scope, key=CONFIG["prompt_registry_auth"]["oauth_client_secret_key"])

@contextmanager
def _sp_auth():
    """Temporarily swap to Service Principal OAuth for prompt registry operations."""
    saved_token = os.environ.pop("DATABRICKS_TOKEN", None)
    os.environ["DATABRICKS_CLIENT_ID"] = _sp_id
    os.environ["DATABRICKS_CLIENT_SECRET"] = _sp_secret
    try:
        yield
    finally:
        os.environ.pop("DATABRICKS_CLIENT_ID", None)
        os.environ.pop("DATABRICKS_CLIENT_SECRET", None)
        if saved_token is not None:
            os.environ["DATABRICKS_TOKEN"] = saved_token

with _sp_auth():
    parent_experiment = mlflow.get_experiment(EXPERIMENT_ID)
    experiment_09 = mlflow.set_experiment(f"{parent_experiment.name}-09-held-out-evaluation")
EXPERIMENT_ID = experiment_09.experiment_id
print(f"Using experiment: {experiment_09.name} (ID: {EXPERIMENT_ID})")
print(f"Aligned judge loaded from: {JUDGE_EXPERIMENT_ID}")

## Step 1: Load Optimized Prompt

Load the best prompt from `@production` (registered by `06-PromptOptimization.ipynb`).

In [ ]:
_eval09_status("prompt", "Loading production prompt")
with _sp_auth():
    prompt_obj = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@production")

optimized_prompt_text = prompt_obj.format()
print(f"Loaded prompt: {PROMPT_NAME} v{prompt_obj.version} (@production)")
print(f"Prompt hash: {hashlib.md5(optimized_prompt_text.encode()).hexdigest()[:12]}")
print(f"Prompt length: {len(optimized_prompt_text)} chars")
print(f"\nFirst 300 chars:\n{optimized_prompt_text[:300]}...")
_eval09_status("prompt", "Loaded production prompt", prompt_version=str(prompt_obj.version), prompt_chars=len(optimized_prompt_text))


## Step 2: Load Aligned Judge

In [ ]:
from mlflow.genai.scorers import get_scorer, Scorer
from mlflow.entities import Feedback
from databricks.sdk import WorkspaceClient
import json as _json
import re as _re
import requests as _requests

GUARDRAIL_SENTINEL = -1.0
_GUARDRAIL_RE = _re.compile(r'(?<![_a-zA-Z])arsenal(?![_a-zA-Z])', _re.IGNORECASE)


def _scrub(text):
    """Replace guardrail-triggering baseball terms in a string."""
    if isinstance(text, str):
        return _GUARDRAIL_RE.sub('pitch repertoire', text)
    return text


try:
    _registered_judge = get_scorer(name=ALIGNED_JUDGE_NAME, experiment_id=JUDGE_EXPERIMENT_ID)
    print(f"Loaded aligned judge metadata: {_registered_judge.name}")
except Exception as exc:
    _registered_judge = None
    print(f"Could not deserialize registered judge metadata; using direct judge fallback: {type(exc).__name__}: {exc}")

ALIGNED_JUDGE_EVAL_MODEL = os.getenv("HELDOUT_JUDGE_MODEL", "databricks:/gpt-5-4-external")
ALIGNED_JUDGE_ENDPOINT = ALIGNED_JUDGE_EVAL_MODEL.replace("databricks:/", "")
_judge_instructions = getattr(_registered_judge, "instructions", None) if _registered_judge is not None else None
if not _judge_instructions:
    _judge_instructions = (
        "Evaluate if the response in {{ outputs }} appropriately analyzes the available data and provides an actionable recommendation "
        "to the question in {{ inputs }}. The response should be accurate, contextually relevant, and give a strategic advantage to the "
        "hitter or coaching staff making the request. Your grading criteria should be: "
        "1: Completely unacceptable. Incorrect data interpretation or no recommendations. "
        "2: Mostly unacceptable. Irrelevant or spurious feedback or weak recommendations with minimal strategic advantage. "
        "3: Somewhat acceptable. Relevant feedback with some strategic advantage. "
        "4: Mostly acceptable. Relevant feedback with strong strategic advantage. "
        "5: Completely acceptable. Relevant feedback with excellent strategic advantage."
    )

_JUDGE_WORKSPACE_CLIENT = WorkspaceClient()
_JUDGE_HOST = _JUDGE_WORKSPACE_CLIENT.config.host.rstrip("/")


def _get_notebook_api_token():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return ctx.apiToken().get()
    except Exception:
        return None


def _get_judge_headers():
    token = _get_notebook_api_token()
    if not token:
        auth = _JUDGE_WORKSPACE_CLIENT.config.authenticate()
        if isinstance(auth, str):
            token = auth
        elif isinstance(auth, dict):
            headers = dict(auth)
            headers.setdefault("Content-Type", "application/json")
            return headers
    if not token:
        raise RuntimeError("Could not resolve a Databricks API token for judge endpoint invocation")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}


class DirectDatabricksJudgeScorer(Scorer):
    """Score with Databricks OpenAI client to avoid MLflow gateway route issues."""

    name: str = ALIGNED_JUDGE_NAME
    instructions: str = _judge_instructions
    model: str = ALIGNED_JUDGE_ENDPOINT

    def __call__(self, *, inputs=None, outputs=None, expectations=None, trace=None):
        if isinstance(outputs, str) and "(Skipped: input guardrail triggered" in outputs:
            return Feedback(name=self.name, value=GUARDRAIL_SENTINEL, rationale="GUARDRAIL_SKIP")

        scrubbed_outputs = _scrub(outputs) if isinstance(outputs, str) else outputs
        user_payload = "outputs: " + _json.dumps(scrubbed_outputs, default=str) + "\ninputs: " + _json.dumps(inputs, default=str)
        system_prompt = self.instructions
        if "{{ outputs }}" in system_prompt or "{{ inputs }}" in system_prompt:
            system_prompt = system_prompt.replace("{{ outputs }}", "the outputs field").replace("{{ inputs }}", "the inputs field")
        system_prompt += (
            "\n\nReturn ONLY a JSON object with fields result and rationale. "
            "The result must be a numeric score from 1 to 5."
        )

        try:
            response = _requests.post(
                f"{_JUDGE_HOST}/serving-endpoints/{self.model}/invocations",
                headers=_get_judge_headers(),
                json={
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_payload},
                    ],
                    "temperature": 0,
                    "max_tokens": 1000,
                    "response_format": {
                        "type": "json_schema",
                        "json_schema": {
                            "name": "ResponseFormat",
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "result": {"type": "number"},
                                    "rationale": {"type": "string"},
                                },
                                "required": ["result", "rationale"],
                            },
                        },
                    },
                },
                timeout=120,
            )
            response.raise_for_status()
            raw = response.json()["choices"][0]["message"]["content"]
            parsed = _json.loads(raw)
            return Feedback(
                name=self.name,
                value=float(parsed["result"]),
                rationale=str(parsed.get("rationale", "")),
            )
        except Exception as e:
            if "guardrail" in str(e).lower() or "input_guardrail_triggered" in str(e):
                logging.warning(f"Guardrail triggered during scoring: {str(e)[:120]}")
                return Feedback(name=self.name, value=GUARDRAIL_SENTINEL, rationale="GUARDRAIL_SKIP")
            raise


aligned_judge = DirectDatabricksJudgeScorer()
print(f"Using direct Databricks judge scorer with endpoint: {ALIGNED_JUDGE_ENDPOINT}")

_eval09_status("judge", "Configured direct Databricks judge scorer", scorer=ALIGNED_JUDGE_NAME, model=ALIGNED_JUDGE_ENDPOINT)


## Step 3: Define Evaluation Helpers

In [ ]:
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=UserWarning, module='pydantic')
logging.getLogger('mlflow.genai.judges.instructions_judge').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.fluent').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.export.mlflow_v3').setLevel(logging.ERROR)
logging.getLogger('mlflow.tracing.provider').setLevel(logging.ERROR)


def _get(item, key, default=""):
    if isinstance(item, dict):
        return item.get(key, default)
    return getattr(item, key, default)


def _extract_compact_response(result) -> str:
    tool_calls = []
    final_text = ""

    for item in result.output:
        item_type = _get(item, "type")

        if item_type == "function_call":
            name = _get(item, "name", "unknown")
            args_str = _get(item, "arguments", "{}")
            if len(args_str) > 300:
                args_str = args_str[:300] + "..."
            tool_calls.append(f"  - {name}({args_str})")

        elif item_type == "message":
            content = _get(item, "content", [])
            if isinstance(content, list):
                for block in content:
                    block_type = _get(block, "type") if isinstance(block, dict) else getattr(block, "type", "")
                    if block_type == "output_text":
                        text = _get(block, "text", "") if isinstance(block, dict) else getattr(block, "text", "")
                        if text:
                            final_text = text
            elif isinstance(content, str):
                final_text = content

        elif item_type == "text":
            text = _get(item, "text", "")
            if text:
                final_text = text

    parts = []
    if tool_calls:
        parts.append("[Tool Calls]\n" + "\n".join(tool_calls))
    if final_text:
        parts.append("[Agent Analysis]\n" + final_text)

    compact = "\n\n".join(parts) if parts else "(no response)"
    return _scrub(compact)



def _maybe_json(value):
    if not isinstance(value, str):
        return value
    text = value.strip()
    if not text:
        return value
    if text[0] not in "[{":
        return value
    try:
        return json.loads(text)
    except Exception:
        return value


def _extract_user_message(value) -> str:
    """Normalize MLflow eval dataset shapes down to the actual user question."""
    value = _maybe_json(value)

    if isinstance(value, dict):
        if "inputs" in value:
            return _extract_user_message(value["inputs"])
        if "input" in value:
            return _extract_user_message(value["input"])
        if "messages" in value:
            return _extract_user_message(value["messages"])
        for key in ("query", "question", "content", "text"):
            if key in value and value[key] is not None:
                return _extract_user_message(value[key])
        return json.dumps(value, default=str)

    if isinstance(value, list):
        for item in value:
            parsed = _maybe_json(item)
            if isinstance(parsed, dict) and parsed.get("role") == "user":
                return _extract_user_message(parsed.get("content", ""))
        if value:
            return _extract_user_message(value[0])
        return ""

    return str(value)


def _normalize_eval_inputs(inputs) -> dict:
    user_message = _extract_user_message(inputs).strip()
    return {"input": [{"role": "user", "content": user_message}]}


def eval_predict_fn_factory(agent_module_name, prompt_text):
    """Create a predict function for mlflow.genai.evaluate().

    Tracing stays enabled so evaluate() captures full traces.
    """
    with _sp_auth():
        mod = importlib.import_module(agent_module_name)
        agent_instance = mod.AGENT

    def predict_fn(input):
        user_message = _extract_user_message(input).strip()
        _eval09_status("predict", "Starting prediction", agent_module=agent_module_name, question=user_message[:250])

        messages = [
            {"role": "system", "content": prompt_text},
            {"role": "user", "content": user_message},
        ]
        try:
            result = agent_instance.predict({"input": messages})
            compact = _extract_compact_response(result)
            _eval09_status("predict", "Finished prediction", agent_module=agent_module_name, question=user_message[:250], response_preview=compact[:500])
            return compact
        except Exception as e:
            err_str = str(e)
            if "input_guardrail_triggered" in err_str or "guardrail" in err_str.lower():
                logging.warning(f"Guardrail triggered during eval: {user_message[:80]}")
                return "(Skipped: input guardrail triggered)"
            _eval09_status("predict", "Prediction failed", agent_module=agent_module_name, question=user_message[:250], error=err_str[:1000])
            raise

    return predict_fn


print("Evaluation helpers defined.")


## Step 4: Load Evaluation Dataset

Load the held-out evaluation dataset created by `04-Evaluation.ipynb`.

In [ ]:
from mlflow.genai.datasets import get_dataset

EVAL_DATASET_NAME = CONFIG["evaluation"]["dataset_name"]
eval_ds = get_dataset(name=EVAL_DATASET_NAME)
eval_df = eval_ds.to_df()

print(f"Dataset columns: {list(eval_df.columns)}")
print(f"First row keys: {eval_df.iloc[0].to_dict().keys()}")


def _infer_query_category(user_message: str) -> str:
    text = (user_message or "").lower()
    if any(term in text for term in ["arsenal", "pitch types", "what pitches", "does he throw"]):
        return "pitcher_arsenal"
    if any(term in text for term in ["runner", "runners", "on second", "on 2nd", "men on", "base"]):
        return "runner_state"
    if any(term in text for term in ["0-0", "0-1", "0-2", "1-0", "1-1", "1-2", "2-0", "2-1", "2-2", "3-0", "3-1", "3-2", "count"]):
        return "count_tendency"
    if any(term in text for term in ["against", "vs", "versus", "pitch to", "approach"]):
        return "direct_matchup"
    if any(term in text for term in ["list", "batters", "roster", "lineup", "available"]):
        return "team_roster"
    if any(term in text for term in ["recommend", "best matchup", "who should", "which hitter"]):
        return "team_matchup_recommendation"
    if any(term in text for term in ["spin rate", "velocity", "release speed", "fastball", "team pitch", "aggregate"]):
        return "genie_sql"
    return "other"


eval_data = []
category_counts = {}
for _, row in eval_df.iterrows():
    inputs = row.get("inputs")
    if inputs is None:
        continue
    if isinstance(inputs, str):
        inputs = _maybe_json(inputs)
    normalized_inputs = _normalize_eval_inputs(inputs)
    user_message = _extract_user_message(normalized_inputs)
    query_category = _infer_query_category(user_message)
    entry = {"inputs": normalized_inputs}

    expectations = row.get("expectations")
    if expectations is not None:
        if isinstance(expectations, str):
            expectations = json.loads(expectations)
        if expectations:
            entry["expectations"] = expectations
    entry.setdefault("expectations", {})["query_category"] = query_category
    category_counts[query_category] = category_counts.get(query_category, 0) + 1

    eval_data.append(entry)

print(f"Loaded {len(eval_data)} evaluation records from dataset '{EVAL_DATASET_NAME}'")
print(f"Category distribution before limit: {category_counts}")
heldout_limit = int(os.getenv("HELDOUT_EVAL_LIMIT", "5"))
if heldout_limit > 0 and len(eval_data) > heldout_limit:
    eval_data = eval_data[:heldout_limit]
    print(f"Using first {len(eval_data)} records for held-out comparison. Set HELDOUT_EVAL_LIMIT=0 for the full dataset.")
else:
    print(f"Using all {len(eval_data)} records for held-out comparison.")
limited_category_counts = {}
for entry in eval_data:
    cat = entry.get("expectations", {}).get("query_category", "unknown")
    limited_category_counts[cat] = limited_category_counts.get(cat, 0) + 1
print(f"Category distribution in eval slice: {limited_category_counts}")
print(f"Sample inputs keys: {list(eval_data[0]['inputs'].keys())}")
has_expectations = any("expectations" in e for e in eval_data)
print(f"Has expectations: {has_expectations}")
if has_expectations:
    print(f"Sample expectations: {eval_data[0].get('expectations', {})}")

_eval09_status("dataset", "Loaded held-out eval data", records=len(eval_data), category_counts=limited_category_counts)


## Step 5: Diagnostic -- test scorer on one example

Run evaluate on a single row to verify the scorer produces metrics before launching the full run.

In [ ]:
_eval09_status("diagnostic", "Starting single-row diagnostic")
from mlflow.genai import evaluate

diag_pfn = eval_predict_fn_factory("agent", optimized_prompt_text)

print("Running single-row diagnostic...")
diag_result = evaluate(
    data=eval_data[:1],
    predict_fn=diag_pfn,
    scorers=[aligned_judge],
)

print("\n" + "=" * 60)
print("FULL RETURN OBJECT INSPECTION")
print("=" * 60)
print(f"Type: {type(diag_result)}")
print(f"Dir:  {[a for a in dir(diag_result) if not a.startswith('_')]}")
for attr in [a for a in dir(diag_result) if not a.startswith('_')]:
    try:
        val = getattr(diag_result, attr)
        if not callable(val):
            val_str = str(val)
            if len(val_str) > 500:
                val_str = val_str[:500] + "..."
            print(f"\n  .{attr} = {val_str}")
    except Exception as e:
        print(f"\n  .{attr} -> ERROR: {e}")

print("\n" + "=" * 60)
print("TRACE INSPECTION")
print("=" * 60)
diag_traces = mlflow.search_traces(run_id=diag_result.run_id)
print(f"Traces found: {len(diag_traces)}")
if len(diag_traces) > 0:
    row = diag_traces.iloc[0]
    print(f"  Columns: {list(diag_traces.columns)}")
    print(f"  Status: {row.get('status', 'N/A')}")
    print(f"  Response preview: {str(row.get('response', ''))[:300]}")
    assessments = row.get("assessments", [])
    print(f"  Assessments: {len(assessments) if assessments else 0}")
    if assessments:
        for a in assessments:
            print(f"    - {a}")

if not diag_result.metrics:
    print("\nWARNING: Metrics are empty. Check details above.")
    print("Continuing to full evaluation anyway.")
else:
    print("\nScorer is working. Proceeding to full evaluation.")
_eval09_status("diagnostic", "Finished single-row diagnostic", run_id=getattr(diag_result, "run_id", None), metrics=getattr(diag_result, "metrics", {}))


## Step 6: Run Full Evaluation

Evaluate both agents on the complete held-out dataset.

In [ ]:
os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"
print("Concurrency: max_workers=1, max_scorer_workers=1")

eval_results = {}
_eval09_status("full_eval", "Starting full held-out evaluation", records=len(eval_data))
for agent_module in ["agent", "agent_with_skills"]:
    label = "base" if agent_module == "agent" else "skills"
    print(f"\n{'=' * 60}")
    print(f"Evaluating: {agent_module} (label: {label})")
    print(f"{'=' * 60}")
    _eval09_status("agent_eval", "Starting agent evaluation", agent_module=agent_module, label=label, records=len(eval_data))

    pfn = eval_predict_fn_factory(agent_module, optimized_prompt_text)

    result = evaluate(
        data=eval_data,
        predict_fn=pfn,
        scorers=[aligned_judge],
    )

    eval_results[label] = result
    print(f"  Metrics: {result.metrics}")
    _eval09_status("agent_eval", "Finished agent evaluation", agent_module=agent_module, label=label, run_id=result.run_id, metrics=result.metrics)

    if hasattr(result, "tables") and result.tables:
        for tname, tdf in result.tables.items():
            print(f"  Table '{tname}': {len(tdf)} rows, columns: {list(tdf.columns)}")

print(f"\n{'=' * 60}")
print("HELD-OUT EVALUATION COMPLETE")
print(f"{'=' * 60}")
_eval09_status("full_eval", "Completed full held-out evaluation", run_ids={k: v.run_id for k, v in eval_results.items()}, metrics={k: v.metrics for k, v in eval_results.items()})


## Step 7: Summarize Results

In [ ]:
import numpy as np

CHECKPOINT_TABLE = f"{CONFIG['workspace']['catalog']}.{CONFIG['workspace']['schema']}.gepa_experiment_checkpoint"


def _extract_scores_from_traces(run_id, scorer_name):
    """Pull per-row scores from trace assessments when result.metrics is empty."""
    traces_df = mlflow.search_traces(run_id=run_id)
    scores = []
    for _, row in traces_df.iterrows():
        for a in (row.get("assessments") or []):
            if a.get("assessment_name") == scorer_name:
                val = a.get("feedback", {}).get("value")
                if val == "yes":
                    scores.append(1.0)
                elif val == "no":
                    scores.append(0.0)
                elif isinstance(val, (int, float)):
                    scores.append(float(val))
    return scores




def _scores_by_category(run_id, scorer_name, eval_records):
    run = mlflow.get_run(run_id)
    traces_df = mlflow.search_traces(run_id=run_id, locations=[run.info.experiment_id])
    traces_df = traces_df.sort_values("request_time") if "request_time" in traces_df.columns else traces_df
    rows = []
    for idx, (_, trace_row) in enumerate(traces_df.iterrows()):
        if idx >= len(eval_records):
            break
        category = eval_records[idx].get("expectations", {}).get("query_category", "unknown")
        score = None
        for assessment in (trace_row.get("assessments") or []):
            if assessment.get("assessment_name") == scorer_name:
                val = assessment.get("feedback", {}).get("value")
                if isinstance(val, (int, float)):
                    score = float(val)
                elif val == "yes":
                    score = 1.0
                elif val == "no":
                    score = 0.0
        if score is not None and score != GUARDRAIL_SENTINEL:
            rows.append({"category": category, "score": score})
    return rows


def _print_category_breakdown(label, run_id):
    rows = _scores_by_category(run_id, ALIGNED_JUDGE_NAME, eval_data)
    if not rows:
        print(f"  {label}: no category scores found")
        return
    grouped = {}
    for row in rows:
        grouped.setdefault(row["category"], []).append(row["score"])
    print(f"  {label} category breakdown:")
    for category in sorted(grouped):
        vals = grouped[category]
        print(f"    {category:30s} mean={np.mean(vals):.2f}, n={len(vals)}, scores={vals}")

print("GEPA OPTIMIZATION RESULTS (from 06-PromptOptimization)")
print("=" * 90)
if spark.catalog.tableExists(CHECKPOINT_TABLE):
    import pandas as pd
    _cp_df = spark.table(CHECKPOINT_TABLE).filter("agent_type = 'base'").toPandas()
    _cp_df["lift"] = _cp_df["final_score"] / _cp_df["initial_score"]
    print(_cp_df[["run_idx", "initial_score", "final_score", "lift"]].to_string(index=False, float_format="%.4f"))
    print(f"\n  Mean initial (1-5): {_cp_df['initial_score'].mean()*5:.2f} +/- {_cp_df['initial_score'].std()*5:.2f}")
    print(f"  Mean final (1-5):   {_cp_df['final_score'].mean()*5:.2f} +/- {_cp_df['final_score'].std()*5:.2f}")
else:
    print(f"  (Checkpoint table {CHECKPOINT_TABLE} not found)")

print(f"\n{'=' * 90}")
print("HELD-OUT EVALUATION (same optimized prompt, both agents)")
print("=" * 90)
for label in ["base", "skills"]:
    if label not in eval_results:
        continue
    result = eval_results[label]
    metrics = result.metrics
    if metrics:
        print(f"  {label:10s}: {metrics}")
    else:
        scores = _extract_scores_from_traces(result.run_id, ALIGNED_JUDGE_NAME)
        valid = [s for s in scores if s != GUARDRAIL_SENTINEL]
        if valid:
            print(f"  {label:10s}: mean={np.mean(valid):.4f}, n={len(valid)}, guardrail_skips={len(scores)-len(valid)}")
        else:
            print(f"  {label:10s}: no valid scores found (raw scores: {scores[:5]})")

print(f"\n{'=' * 90}")
print("CATEGORY BREAKDOWN")
print("=" * 90)
for label in ["base", "skills"]:
    if label in eval_results:
        _print_category_breakdown(label, eval_results[label].run_id)

print(f"\n{'=' * 90}")
print("SUMMARY FOR PAPER")
print("=" * 90)
if spark.catalog.tableExists(CHECKPOINT_TABLE):
    print(f"\n1) Pre-optimization baseline (n={len(_cp_df)}):")
    print(f"   Score (1-5): {_cp_df['initial_score'].mean()*5:.2f} +/- {_cp_df['initial_score'].std()*5:.2f}")
    print(f"\n2) Align + GEPA (n={len(_cp_df)}):")
    print(f"   Score (1-5): {_cp_df['final_score'].mean()*5:.2f} +/- {_cp_df['final_score'].std()*5:.2f}")
    gain = ((_cp_df['final_score'].mean() - _cp_df['initial_score'].mean()) / _cp_df['initial_score'].mean()) * 100
    print(f"   Gain: {gain:.1f}%")

print(f"\n3) Held-out evaluation (prompt v{prompt_obj.version}, {EVAL_DATASET_NAME} dataset):")
for label in ["base", "skills"]:
    if label not in eval_results:
        continue
    result = eval_results[label]
    if result.metrics:
        print(f"   {label}: {result.metrics}")
    else:
        scores = _extract_scores_from_traces(result.run_id, ALIGNED_JUDGE_NAME)
        valid = [s for s in scores if s != GUARDRAIL_SENTINEL]
        if valid:
            print(f"   {label}: mean={np.mean(valid):.4f} +/- {np.std(valid):.4f} (n={len(valid)}, skips={len(scores)-len(valid)})")
        else:
            print(f"   {label}: no valid scores")

print(f"\nFull traces and per-example scores are in MLflow experiment: {experiment_09.name}")


Latest redacted retest status (2026-05-18):
- Base-agent diagnostic completed: mean 4.00 over 1 trace.
- Most complete recreated base-agent run completed: mean 3.10 over 20 traces.
- Most complete recreated skills-agent run failed before completion: partial mean 3.16 over 19 scored traces.
- Prior comparable run: base mean 1.80 over 20 traces; skills mean 2.45 over 20 traces.
- Conclusion: recreated model path showed better partial performance, but required full held-out base-vs-skills retest was not achieved.


## Held-Out Score Comparison Chart

Visualize aligned-judge scores across the three evaluation configurations:
1. **Baseline** - Original agent with optimized prompt
2. **Optimized Prompt** - GEPA-optimized prompt only
3. **Optimized Prompt + Skills** - GEPA-optimized prompt with generated agent skills

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import mlflow

CONFIG = json.loads(Path("config/atbat_assistant.json").read_text())
ALIGNED_JUDGE_NAME = CONFIG["judges"]["aligned_judge_name"]
GUARDRAIL_SENTINEL = -1.0

if "eval_results" not in globals():
    raise RuntimeError("eval_results not found. Run the evaluation cells before rendering this chart.")

RUN_IDS = {
    "Optimized Prompt": eval_results["base"].run_id,
    "Optimized Prompt + Skills": eval_results["skills"].run_id,
}

SCORER_NAME = ALIGNED_JUDGE_NAME


def _extract_scores_from_run(run_id, scorer_name):
    """Pull per-row numeric scores from trace assessments for a given run."""
    experiment_id = mlflow.get_run(run_id).info.experiment_id
    traces_df = mlflow.search_traces(run_id=run_id, locations=[experiment_id])
    scores = []
    for _, row in traces_df.iterrows():
        for a in (row.get("assessments") or []):
            if a.get("assessment_name") == scorer_name:
                val = a.get("feedback", {}).get("value")
                if val == "yes":
                    scores.append(1.0)
                elif val == "no":
                    scores.append(0.0)
                elif isinstance(val, (int, float)):
                    scores.append(float(val))
    return scores


scores_by_config = {}
for label, run_id in RUN_IDS.items():
    raw = _extract_scores_from_run(run_id, SCORER_NAME)
    valid = [s for s in raw if s != GUARDRAIL_SENTINEL]
    scores_by_config[label] = valid
    print(f"{label}: mean={np.mean(valid):.2f}, std={np.std(valid):.2f}, n={len(valid)}, skipped={len(raw)-len(valid)}")

labels = list(scores_by_config.keys())
means = [np.mean(v) for v in scores_by_config.values()]
stds = [np.std(v) for v in scores_by_config.values()]
counts = [len(v) for v in scores_by_config.values()]

colors = ["#3B82F6", "#10B981"]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, yerr=stds, capsize=6, color=colors, edgecolor="white", linewidth=1.2, width=0.55)

for bar, mean, std, n in zip(bars, means, stds, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.06,
        f"{mean:.2f}\n(n={n})",
        ha="center", va="bottom", fontsize=11, fontweight="bold",
    )

ax.set_ylabel("Aligned Judge Score (1-5)", fontsize=12)
ax.set_title("Held-Out Evaluation: Aligned Judge Scores by Configuration", fontsize=13, fontweight="bold")
ax.set_ylim(0, 5.0)
ax.yaxis.set_major_locator(ticker.MultipleLocator(1.0))
ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
ax.grid(axis="y", alpha=0.3, linestyle="--")
ax.set_axisbelow(True)

baseline_mean = means[0]
for i, (bar, mean) in enumerate(zip(bars, means)):
    if i > 0 and baseline_mean > 0:
        pct_gain = ((mean - baseline_mean) / baseline_mean) * 100
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            0.15,
            f"+{pct_gain:.1f}%",
            ha="center", va="bottom", fontsize=10, color="white", fontweight="bold",
        )

plt.tight_layout()
plt.show()